In [14]:
# notebooks/05_challenger_models.ipynb
import sys
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import brier_score_loss

# 1. Framework Path Setup
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_validation_splits
from src.metrics import calculate_gini_and_ks
from src.scorecard_transformer import ScorecardTransformer

print("=== RUNNING PHASE 5: CHALLENGER FLEET INTEGRITY BENCHMARK ===")

# 2. Ingest Data Splits from Disk Layer
df_train_tweaked, df_val_final, df_oot_final, df_scorecard_points = load_validation_splits()

# Clean execution layer using our Phase 0 central reusable asset
transformer = ScorecardTransformer(df_scorecard_points)
print("Transforming split layers using centralized ScorecardTransformer engine...")
X_train, _ = transformer.fit_transform_portfolio(df_train_tweaked)
y_train = df_train_tweaked['bad'].astype(int)

X_oot, _ = transformer.fit_transform_portfolio(df_oot_final)
y_oot = df_oot_final['bad'].astype(int)

# =====================================================================
# PART A: FLEET MANAGEMENT ENGINE (PRESERVED & INTEGRATED)
# =====================================================================

def execute_challenger_fleet(X_train, y_train, X_oot, y_oot, feature_names):
    """
    Trains and returns predicted probabilities for the complete challenger suite 
    (Unconstrained Random Forest Benchmark vs. Monotone Constrained XGBoost).
    """
    print("\n🚀 Training Unconstrained Random Forest Benchmark...")
    rf_model = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    rf_probs = rf_model.predict_proba(X_oot)[:, 1]
    
    print("🚀 Training Monotone Constrained XGBoost Challenger...")
    # Your explicit business rules logic checking feature name keywords (Preserved)
    constraints = tuple([
        1 if 'dti' in col.lower() or 'inq' in col.lower() 
        else (-1 if 'fico' in col.lower() or 'inc' in col.lower() else 0) 
        for col in feature_names
    ])
    
    xgb_model = XGBClassifier(
        n_estimators=200, 
        max_depth=4, 
        learning_rate=0.05, 
        monotone_constraints=constraints, 
        random_state=42,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    xgb_probs = xgb_model.predict_proba(X_oot)[:, 1]
    
    print("✅ Challenger Fleet Evaluation Run Complete.")
    return {
        "Random_Forest_Probs": rf_probs,
        "XGB_Monotone_Probs": xgb_probs
    }

# Execute the fleet calculations
fleet_outputs = execute_challenger_fleet(X_train, y_train, X_oot, y_oot, transformer.final_features)

# Base production benchmark line
champ_model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
champ_model.fit(X_train, y_train)
champ_probs = champ_model.predict_proba(X_oot)[:, 1]

# Compile probabilities vectors
models_probabilities = {
    "Champion Scorecard (LogReg)": champ_probs,
    "Challenger Fleet (Random Forest)": fleet_outputs["Random_Forest_Probs"],
    "Challenger Fleet (Monotone XGB)": fleet_outputs["XGB_Monotone_Probs"]
}

# =====================================================================
# PART B: GLOBAL SUMMARY MATRIX
# =====================================================================
print("\n=== GENERATING COMPREHENSIVE CHAMPION VS CHALLENGER MATRIX ===")

comparison_ledger = []
for name, probs in models_probabilities.items():
    metrics = calculate_gini_and_ks(y_oot, probs)
    brier_err = brier_score_loss(y_oot, probs)
    
    comparison_ledger.append({
        "Model Architecture": name,
        "Global Gini (%)": round(metrics["Gini"] * 100, 2),
        "Global KS (%)": round(metrics["KS"] * 100, 2),
        "Brier Error Score": round(brier_err, 5)
    })

df_matrix_summary = pd.DataFrame(comparison_ledger)
print(df_matrix_summary.to_string(index=False))

# =====================================================================
# PART C: THE ALL-SEGMENT BATTLE GRID (MRM CRITICAL RECONSTRUCTION)
# =====================================================================
print("\n=== STRESS TEST INSIDE STRATEGIC SUB-SEGMENTS MASTER GRID ===")

# Explicit segments list to build out the full comparison table
target_slices = [
    ('grade', 'A'), ('grade', 'B'), ('grade', 'C'), ('grade', 'D'),
    ('home_ownership', 'MORTGAGE'), ('home_ownership', 'RENT')
]

grid_records = []

for col, val in target_slices:
    is_segment = (df_oot_final[col] == val).values
    y_oot_slice = y_oot[is_segment]
    
    record = {"Segment": f"{col.upper()}: {val}"}
    
    for model_name, probs in models_probabilities.items():
        probs_slice = probs[is_segment]
        metrics_slice = calculate_gini_and_ks(y_oot_slice, probs_slice)
        record[model_name] = round(metrics_slice['Gini'] * 100, 2)
        
    grid_records.append(record)

df_battle_grid = pd.DataFrame(grid_records)
print(df_battle_grid.to_string(index=False))

# Calculate explicit lift advantages
print("\n=== MODEL RISK ARCHITECTURE ASSESSMENT ===")
df_battle_grid['XGB Lift vs Scorecard'] = df_battle_grid['Challenger Fleet (Monotone XGB)'] - df_battle_grid['Champion Scorecard (LogReg)']

max_lift_row = df_battle_grid.loc[df_battle_grid['XGB Lift vs Scorecard'].idxmax()]
print(f"💡 Governance Insight: Complex non-linear models achieve their highest marginal separation lift in [{max_lift_row['Segment']}] with an improvement of +{max_lift_row['XGB Lift vs Scorecard']:.2f}% Gini.")

=== RUNNING PHASE 5: CHALLENGER FLEET INTEGRITY BENCHMARK ===
Transforming split layers using centralized ScorecardTransformer engine...

🚀 Training Unconstrained Random Forest Benchmark...
🚀 Training Monotone Constrained XGBoost Challenger...
✅ Challenger Fleet Evaluation Run Complete.

=== GENERATING COMPREHENSIVE CHAMPION VS CHALLENGER MATRIX ===
              Model Architecture  Global Gini (%)  Global KS (%)  Brier Error Score
     Champion Scorecard (LogReg)            39.96          28.82            0.16400
Challenger Fleet (Random Forest)            38.44          27.91            0.16551
 Challenger Fleet (Monotone XGB)            39.95          29.02            0.16379

=== STRESS TEST INSIDE STRATEGIC SUB-SEGMENTS MASTER GRID ===
                 Segment  Champion Scorecard (LogReg)  Challenger Fleet (Random Forest)  Challenger Fleet (Monotone XGB)
                GRADE: A                        32.01                             32.24                            33.23
       